In [1]:
# =====================================================================
# LAB 1: NLU BANCÁRIO COM REGRESSÃO LOGÍSTICA
# =====================================================================

import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

dados_treino = [
    ("quero fazer um pix", "pix"),
    ("como faço uma transferência", "transferencia"),
    ("quero consultar meu saldo", "saldo"),
    ("qual é o meu saldo", "saldo"),
    ("quero pagar uma conta", "pagamento"),
    ("preciso pagar um boleto", "pagamento"),
    ("quero sacar dinheiro", "saque"),
    ("preciso sacar dinheiro", "saque"),
    ("quero depositar dinheiro", "deposito"),
    ("fiz um pix errado", "pix"),
    ("como faço um pix de 100 reais", "pix"),
    ("quero transferir 500 reais", "transferencia"),
    ("meu saldo está errado", "saldo"),
    ("quero pagar 200 reais", "pagamento"),
    ("quero sacar 50 reais", "saque"),
    ("quero depositar 300 reais", "deposito"),
    ("oi tudo bem", "fora_escopo"),
    ("qual é a previsão do tempo", "fora_escopo"),
    ("me conte uma piada", "fora_escopo"),
    ("quem ganhou o jogo", "fora_escopo")
]

stopwords = ["o", "a", "os", "as", "de", "do", "da", "dos", "das",
             "um", "uma", "e", "é", "em", "meu", "minha", "como",
             "quero", "preciso"]

# 1. PRÉ-PROCESSAMENTO
def pré_processar(texto):
    texto_limpo = re.sub(r'[^\w\s]', '', texto.lower()).strip()
    tokens = texto_limpo.split()

    # TODO 1.1: Filtre as stopwords da lista 'tokens' usando List Comprehension
    tokens_filtrados = [token for token in tokens if token not in stopwords]

    return " ".join(tokens_filtrados)

X_treino_limpo = [pré_processar(item[0]) for item in dados_treino]
y_treino = [item[1] for item in dados_treino]

# 2. VETORIZAÇÃO E TREINAMENTO
vectorizer = TfidfVectorizer()

# TODO 1.2: Aprenda o vocabulário e vetorize 'X_treino_limpo' em um único passo
X_vetorizado = vectorizer.fit_transform(X_treino_limpo)

modelo = LogisticRegression()
modelo.fit(X_vetorizado, y_treino)

# 3. EXTRAÇÃO DE ENTIDADE (REGEX)
def extrair_valor_reais(texto_original):
    match = re.search(r'(\d+)\s*reais', texto_original, re.IGNORECASE)

    # TODO 1.3: Se houver match, converta o grupo 1 para float. Caso contrário, retorne None.
    return float(match.group(1)) if match else None

# 4. PIPELINE NLU
def processar_nlu(mensagem_usuario, threshold=0.55):
    texto_p = pré_processar(mensagem_usuario)
    vetor_input = vectorizer.transform([texto_p])

    probas = modelo.predict_proba(vetor_input)[0]

    # TODO 1.4: Extraia a MAIOR probabilidade (max) e a intenção prevista (argmax em modelo.classes_)
    maior_confianca = max(probas)
    intencao_prevista = modelo.classes_[probas.argmax()]

    # TODO 1.5: Se maior_confianca < threshold ou intencao == "fora_escopo", retorne FALLBACK.
    # Caso contrário, retorne SUCESSO com a intenção, confiança e o valor extraído.
    if maior_confianca < threshold or intencao_prevista == "fora_escopo":
        return {
            "status": "FALLBACK"
        }
    else:
        return {
            "status": "SUCESSO",
            "intencao": intencao_prevista,
            "confianca": maior_confianca,
            "valor": extrair_valor_reais(mensagem_usuario)
        }

# --- TESTE 1 ---
if __name__ == "__main__":
    print(processar_nlu("quero fazer um pix de 100 reais"))

{'status': 'FALLBACK'}
